In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Loading Ames Housing dataset...")

# Read the data from the Excel file
df = pd.read_excel('AmesHousing.xlsx')

# Select only numerical columns and drop columns with more than 20% missing values
numeric_df = df.select_dtypes(include=[np.number]).dropna(axis=1, thresh=len(df)*0.8)

# Fill the remaining missing values with the mean of their respective columns
numeric_df = numeric_df.fillna(numeric_df.mean())

# Target variable (What we want to predict: Sale Price)
y = numeric_df['SalePrice']

# Option 1: ALL NUMERICAL FEATURES
# Dropping columns that are not useful for prediction like Price itself, ID, and Order
X_all = numeric_df.drop(['SalePrice', 'Order', 'PID'], axis=1, errors='ignore')

# Option 2: SUBSET OF FEATURES - Selecting logical and highly impactful features
desired_features = ['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Total Bsmt SF', 'Year Built', '1st Flr SF']

# SMART FILTER: Sadece senin Excel dosyasında gerçekten var olan özellikleri seçer, hata vermez!
strong_features = [f for f in desired_features if f in numeric_df.columns]
X_sub = numeric_df[strong_features]

print(f"✅ Data ready! All Features shape: {X_all.shape} | Subset Features shape: {X_sub.shape}")
print(f"🔍 Selected Subset Features: {strong_features}")

Loading Ames Housing dataset...
✅ Data ready! All Features shape: (2930, 8) | Subset Features shape: (2930, 4)
🔍 Selected Subset Features: ['Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Year Built']


In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, r2_score
from IPython.display import display

# Experiment Configurations for Ames Housing Neural Network
experiments = [
    {'id': 1, 'features': 'All', 'val_split': 0.2, 'layers': 1, 'nodes': [64], 'epochs': 50, 'batch': 32, 'lr': 0.01, 'note': 'Baseline (All Features)'},
    {'id': 2, 'features': 'Subset', 'val_split': 0.2, 'layers': 1, 'nodes': [64], 'epochs': 50, 'batch': 32, 'lr': 0.01, 'note': 'Baseline (Subset)'},
    {'id': 3, 'features': 'Subset', 'val_split': 0.1, 'layers': 2, 'nodes': [128, 64], 'epochs': 100, 'batch': 32, 'lr': 0.001, 'note': 'Val Size 10%, 2 Layers'},
    {'id': 4, 'features': 'Subset', 'val_split': 0.3, 'layers': 2, 'nodes': [128, 64], 'epochs': 100, 'batch': 32, 'lr': 0.001, 'note': 'Val Size 30%, 2 Layers'},
    {'id': 5, 'features': 'All', 'val_split': 0.2, 'layers': 3, 'nodes': [256, 128, 64], 'epochs': 150, 'batch': 32, 'lr': 0.001, 'note': 'Deep Net (All Features)'},
    {'id': 6, 'features': 'Subset', 'val_split': 0.2, 'layers': 3, 'nodes': [256, 128, 64], 'epochs': 150, 'batch': 32, 'lr': 0.001, 'note': 'Deep Net (Subset)'},
    {'id': 7, 'features': 'Subset', 'val_split': 0.2, 'layers': 2, 'nodes': [128, 64], 'epochs': 200, 'batch': 64, 'lr': 0.001, 'note': 'More Epochs, Large Batch'},
    {'id': 8, 'features': 'All', 'val_split': 0.2, 'layers': 2, 'nodes': [128, 64], 'epochs': 100, 'batch': 32, 'lr': 0.0005, 'note': 'Low Learning Rate'}
]

results = []
print("Experiments started\n")

for exp in experiments:
    # 1. Select the dataset based on the experiment configuration
    X = X_all if exp['features'] == 'All' else X_sub
    
    # 2. Train-Test Split (80% Train, 20% Test)
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 3. Data Scaling - Crucial for Neural Networks in regression tasks
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_full)
    X_test_scaled = scaler.transform(X_test)
    
    # 4. Build the Model
    model = Sequential()
    for i in range(exp['layers']):
        if i == 0:
            model.add(Dense(exp['nodes'][i], activation='relu', input_dim=X_train_scaled.shape[1]))
        else:
            model.add(Dense(exp['nodes'][i], activation='relu'))
            
    # CAUTION: Since this is regression (predicting a continuous price), the output layer has no activation function!
    model.add(Dense(1)) 
    
    # 5. Compile the Model (Loss: Mean Squared Error - standard for regression)
    optimizer = Adam(learning_rate=exp['lr'])
    model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])
    
    # 6. Start Training (Validation split is applied here as requested by the instructor)
    model.fit(X_train_scaled, y_train_full, 
              validation_split=exp['val_split'], 
              epochs=exp['epochs'], 
              batch_size=exp['batch'], 
              verbose=0)
    
    # 7. Evaluate with Test Set
    y_pred = model.predict(X_test_scaled, verbose=0)
    mae = mean_absolute_error(y_test, y_pred) # Mean Absolute Error in Dollars
    r2 = r2_score(y_test, y_pred)             # R-Squared Accuracy Score
    
    # Save the results
    results.append({
        'Experiment ID': exp['id'],
        'Features Used': exp['features'],
        'Val Set Size': f"{int(exp['val_split']*100)}%",
        'Hidden Layers': exp['layers'],
        'Nodes': str(exp['nodes']),
        'Epochs': exp['epochs'],
        'Batch Size': exp['batch'],
        'Learning Rate': exp['lr'],
        'MAE Error ($)': round(mae, 2),
        'R2 Score': round(r2, 4),
        'Description': exp['note']
    })
    print(f"Experiment {exp['id']} completed -> Error (MAE): ${mae:,.0f} | Accuracy (R2): {r2:.2f}")

# Create and display the results table
df_results = pd.DataFrame(results)
df_results.to_csv('AmesHousing_NN_Experiment_Results.csv', index=False, sep=';')
print("\n🎉 ALL EXPERIMENTS COMPLETED! Results saved.")
display(df_results)

Experiments started



c:\Users\serza\Documents\GitHub\DEAI-Serzat\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Experiment 1 completed -> Error (MAE): $59,630 | Accuracy (R2): 0.31


c:\Users\serza\Documents\GitHub\DEAI-Serzat\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Experiment 2 completed -> Error (MAE): $83,993 | Accuracy (R2): -0.13


...

c:\Users\serza\Documents\GitHub\DEAI-Serzat\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Experiment 7 completed -> Error (MAE): $25,550 | Accuracy (R2): 0.81


c:\Users\serza\Documents\GitHub\DEAI-Serzat\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Experiment 8 completed -> Error (MAE): $41,994 | Accuracy (R2): 0.57

🎉 ALL EXPERIMENTS COMPLETED! Results saved.


,Experiment ID,Features Used,Val Set Size,Hidden Layers,Nodes,Epochs,Batch Size,Learning Rate,MAE Error ($),R2 Score,Description
0,1,All,20%,1,[64],50,32,0.0100,59629.80,0.3057,Baseline (All Features)
1,2,Subset,20%,1,[64],50,32,0.0100,83993.02,-0.1340,Baseline (Subset)
2,3,Subset,10%,2,"[128, 64]",100,32,0.0010,26294.92,0.8013,"Val Size 10%, 2 Layers"
3,4,Subset,30%,2,"[128, 64]",100,32,0.0010,26393.38,0.7925,"Val Size 30%, 2 Layers"
4,5,All,20%,3,"[256, 128, 64]",150,32,0.0010,22696.42,0.8248,Deep Net (All Features)
5,6,Subset,20%,3,"[256, 128, 64]",150,32,0.0010,23069.63,0.8265,Deep Net (Subset)
6,7,Subset,20%,2,"[128, 64]",200,64,0.0010,25549.89,0.8062,"More Epochs, Large Batch"
7,8,All,20%,2,"[128, 64]",100,32,0.0005,41993.77,0.5717,Low Learning Rate
